In [0]:
from pyspark.sql import functions as F

raw_path = "/Volumes/workspace/default/australiya_raw_data"

# Load all yearly AEMO electricity files together
elec_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "Australia_AEMO*.csv")
    .csv(raw_path)
)

elec_raw.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.bronze_electricity_load"
)

print(f"Loaded {elec_raw.count()} rows | Columns: {elec_raw.columns}")
elec_raw.show(5, truncate=False)

Loaded 661728 rows | Columns: ['Timestamp', 'NSW1', 'QLD1', 'SA1', 'TAS1', 'VIC1']
+-------------------+-------+-------+-------+-------+-------+
|Timestamp          |NSW1   |QLD1   |SA1    |TAS1   |VIC1   |
+-------------------+-------+-------+-------+-------+-------+
|2024-01-01 00:00:00|6587.63|6294.38|1296.67|1035.11|4024.55|
|2024-01-01 00:05:00|6574.92|6228.31|1293.98|1033.29|3977.1 |
|2024-01-01 00:10:00|6651.09|6225.45|1300.29|1043.26|3982.35|
|2024-01-01 00:15:00|6538.96|6233.57|1302.81|1041.05|3965.75|
|2024-01-01 00:20:00|6497.99|6189.11|1311.46|1035.57|3932.93|
+-------------------+-------+-------+-------+-------+-------+
only showing top 5 rows


In [0]:
import requests
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

elec_df = spark.table("workspace.default.bronze_electricity_load")
time_col = elec_df.columns[0]
regions = elec_df.columns[1:]

# Find the start and end dates from the electricity dataset
bounds = elec_df.select(
    F.min(F.to_date(F.col(time_col))).alias("start"),
    F.max(F.to_date(F.col(time_col))).alias("end")
).collect()[0]

start_dt, end_dt = str(bounds["start"]), str(bounds["end"])
print(f"Date range: {start_dt} to {end_dt}")

# Regional lookup: State, Lat, Lon, Pop (M), Avg Tariff, Peak Tariff, Industrial %
STATE_INFO = {
    "NSW": ("New South Wales", -33.8688, 151.2093, 8.34, 0.34, 0.48, 32.0),
    "QLD": ("Queensland", -27.4698, 153.0251, 5.46, 0.31, 0.44, 38.0),
    "VIC": ("Victoria", -37.8136, 144.9631, 6.81, 0.29, 0.42, 34.0),
    "SA":  ("South Australia", -34.9285, 138.6007, 1.85, 0.41, 0.56, 26.0),
    "TAS": ("Tasmania", -42.8821, 147.3272, 0.57, 0.27, 0.36, 45.0),
    "WA":  ("Western Australia", -31.9505, 115.8605, 2.88, 0.32, 0.45, 42.0),
    "SNOWY": ("Snowy Region", -36.2333, 148.6167, 0.20, 0.30, 0.42, 20.0),
}

def lookup_region(col_name):
    upper = col_name.upper()
    for key, val in STATE_INFO.items():
        if key in upper:
            return val
    return (col_name, -33.8688, 151.2093, 2.50, 0.32, 0.45, 30.0)

# Build and save demographics & tariff table
demo_records = []
for reg in regions:
    name, lat, lon, pop, avg_t, peak_t, ind = lookup_region(reg)
    demo_records.append({
        "Region": reg, "State_Name": name, "Latitude": lat, "Longitude": lon,
        "Population_Millions": pop, "Avg_Tariff_AUD_kWh": avg_t,
        "Peak_Tariff_AUD_kWh": peak_t, "Industrial_Share_Pct": ind
    })

spark.createDataFrame(pd.DataFrame(demo_records)).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.bronze_demographics_tariffs")

# Pull daily weather from Open-Meteo (with local seasonal fallback if blocked)
weather_list = []
date_seq = pd.date_range(start=start_dt, end=end_dt, freq="D")

for item in demo_records:
    reg, lat, lon = item["Region"], item["Latitude"], item["Longitude"]
    got_api_data = False
    try:
        resp = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat, "longitude": lon,
                "start_date": start_dt, "end_date": end_dt,
                "daily": "temperature_2m_mean,temperature_2m_max,precipitation_sum",
                "timezone": "auto"
            },
            timeout=10
        )
        if resp.status_code == 200 and "daily" in resp.json():
            daily = resp.json()["daily"]
            weather_list.append(pd.DataFrame({
                "Date": pd.to_datetime(daily["time"]).date,
                "Region": reg,
                "Temp_Mean_C": daily["temperature_2m_mean"],
                "Temp_Max_C": daily["temperature_2m_max"],
                "Precipitation_mm": daily["precipitation_sum"]
            }))
            got_api_data = True
    except Exception:
        pass

    if not got_api_data:
        rng = np.random.default_rng(abs(hash(reg)) % 10000)
        doy = date_seq.dayofyear.values
        avg_base = 24.0 + (lat + 27.0) * 0.65
        t_mean = np.round(avg_base + 6.5 * np.cos(2 * np.pi * (doy - 20) / 365.25) + rng.normal(0, 2.5, len(date_seq)), 2)
        t_max = np.round(t_mean + rng.uniform(3.5, 8.5, len(date_seq)), 2)
        rain = np.round(np.where(rng.random(len(date_seq)) < 0.25, rng.exponential(6.0, len(date_seq)), 0.0), 2)
        weather_list.append(pd.DataFrame({
            "Date": date_seq.date, "Region": reg,
            "Temp_Mean_C": t_mean, "Temp_Max_C": t_max, "Precipitation_mm": rain
        }))

weather_pdf = pd.concat(weather_list, ignore_index=True)
spark.createDataFrame(weather_pdf).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.bronze_weather")

print(f"Weather rows saved: {len(weather_pdf)}")

Date range: 2006-01-01 to 2025-06-01
Weather rows saved: 35460


In [0]:
from pyspark.sql import functions as F

elec = spark.table("workspace.default.bronze_electricity_load")
weather = spark.table("workspace.default.bronze_weather")
demo = spark.table("workspace.default.bronze_demographics_tariffs")

time_col = elec.columns[0]
regions = elec.columns[1:]

# Convert wide regional columns into rows using Spark SQL stack()
pairs = ", ".join([f"'{r}', cast(`{r}` as double)" for r in regions])
unpivot_sql = f"stack({len(regions)}, {pairs}) as (Region, Load_MW)"

cleaned = (
    elec
    .select(F.to_timestamp(F.col(time_col)).alias("Timestamp"), F.expr(unpivot_sql))
    .dropna(subset=["Timestamp", "Load_MW"])
    .filter(F.col("Load_MW") > 0)
    .dropDuplicates(["Timestamp", "Region"])
)

# Add time breakdown, Australian seasons, and peak-hour buckets
enriched = (
    cleaned
    .withColumn("Date", F.to_date("Timestamp"))
    .withColumn("Year", F.year("Timestamp"))
    .withColumn("Month", F.month("Timestamp"))
    .withColumn("Hour", F.hour("Timestamp"))
    .withColumn("DayOfWeek", F.dayofweek("Timestamp"))
    .withColumn("IsWeekend", F.when(F.col("DayOfWeek").isin([1, 7]), "Weekend").otherwise("Weekday"))
    .withColumn(
        "Season",
        F.when(F.col("Month").isin([12, 1, 2]), "Summer")
         .when(F.col("Month").isin([3, 4, 5]), "Autumn")
         .when(F.col("Month").isin([6, 7, 8]), "Winter")
         .otherwise("Spring")
    )
    .withColumn(
        "TimePeriod",
        F.when((F.col("Hour") >= 16) & (F.col("Hour") <= 21), "Peak (4PM-9PM)")
         .when((F.col("Hour") >= 7) & (F.col("Hour") < 16), "Daytime Shoulder")
         .otherwise("Off-Peak Night")
    )
)

# Combine electricity load with weather and economic context
silver_df = (
    enriched
    .join(weather, on=["Date", "Region"], how="left")
    .join(demo, on="Region", how="left")
    .withColumn(
        "Applied_Tariff_AUD_kWh",
        F.when(F.col("TimePeriod") == "Peak (4PM-9PM)", F.col("Peak_Tariff_AUD_kWh"))
         .otherwise(F.col("Avg_Tariff_AUD_kWh"))
    )
    .withColumn("Load_Per_Capita_kW", F.round((F.col("Load_MW") * 1000.0) / (F.col("Population_Millions") * 1e6), 4))
    .withColumn("Est_Hourly_Cost_AUD", F.round(F.col("Load_MW") * 1000.0 * F.col("Applied_Tariff_AUD_kWh"), 2))
)

silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.silver_smart_electricity"
)

print("Silver rows:", silver_df.count())

Silver rows: 3308213


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.default.silver_smart_electricity")

# 1. Monthly regional summary for long-term trends
monthly_summary = (
    silver.groupBy("Year", "Month", "Region", "State_Name", "Season", "Population_Millions", "Industrial_Share_Pct")
    .agg(
        F.round(F.avg("Load_MW"), 2).alias("Avg_Load_MW"),
        F.round(F.max("Load_MW"), 2).alias("Peak_Load_MW"),
        F.round(F.min("Load_MW"), 2).alias("Min_Load_MW"),
        F.round(F.avg("Load_Per_Capita_kW"), 4).alias("Avg_Load_Per_Capita_kW"),
        F.round(F.avg("Temp_Mean_C"), 2).alias("Avg_Temp_C"),
        F.round(F.max("Temp_Max_C"), 2).alias("Max_Temp_C"),
        F.round(F.avg("Est_Hourly_Cost_AUD") / 1e6, 3).alias("Avg_Hourly_Grid_Value_M_AUD")
    )
    .withColumn("YearMonth", F.to_date(F.concat_ws("-", F.col("Year"), F.lpad(F.col("Month"), 2, "0"), F.lit("01"))))
)
monthly_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_regional_monthly_summary"
)

# 2. 24-hour consumption profile
hourly_profile = (
    silver.groupBy("Region", "State_Name", "Season", "IsWeekend", "Hour", "TimePeriod")
    .agg(
        F.round(F.avg("Load_MW"), 2).alias("Avg_Load_MW"),
        F.round(F.max("Load_MW"), 2).alias("Max_Peak_MW"),
        F.round(F.avg("Load_Per_Capita_kW"), 4).alias("Avg_Load_Per_Capita_kW")
    )
)
hourly_profile.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_hourly_demand_profile"
)

# 3. Daily peak demand + weather table (also used for ML)
daily_peaks = (
    silver.groupBy("Date", "Year", "Month", "DayOfWeek", "IsWeekend", "Season", "Region", "State_Name", "Population_Millions")
    .agg(
        F.round(F.avg("Load_MW"), 2).alias("Daily_Avg_Load_MW"),
        F.round(F.max("Load_MW"), 2).alias("Daily_Peak_Load_MW"),
        F.round(F.min("Load_MW"), 2).alias("Daily_Min_Load_MW"),
        F.round(F.avg("Temp_Mean_C"), 2).alias("Temp_Mean_C"),
        F.round(F.max("Temp_Max_C"), 2).alias("Temp_Max_C"),
        F.round(F.avg("Precipitation_mm"), 2).alias("Precipitation_mm")
    )
    .withColumn("Peak_to_Avg_Ratio", F.round(F.col("Daily_Peak_Load_MW") / F.col("Daily_Avg_Load_MW"), 3))
)
daily_peaks.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_daily_peak_demand"
)

print("Gold tables ready.")

Gold tables ready.


In [0]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = spark.table("workspace.default.gold_daily_peak_demand").dropna().toPandas()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Region", "Date"]).reset_index(drop=True)

# Engineer calendar and lag features for time-series forecasting
df["DayOfYear"] = df["Date"].dt.dayofyear
df["IsWeekend_Num"] = (df["IsWeekend"] == "Weekend").astype(int)
df["Region_Code"] = df["Region"].astype("category").cat.codes
df["Lag_1d_Peak_MW"] = df.groupby("Region")["Daily_Peak_Load_MW"].shift(1)
df["Rolling_7d_Peak_MW"] = df.groupby("Region")["Daily_Peak_Load_MW"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)
df = df.dropna().reset_index(drop=True)

feature_cols = [
    "Region_Code", "Year", "Month", "DayOfYear", "DayOfWeek", "IsWeekend_Num",
    "Temp_Mean_C", "Temp_Max_C", "Precipitation_mm", "Population_Millions",
    "Lag_1d_Peak_MW", "Rolling_7d_Peak_MW"
]
target_col = "Daily_Peak_Load_MW"

# Time-based split: hold out the most recent 2 years for testing
cutoff_year = int(df["Year"].max()) - 1
train = df[df["Year"] < cutoff_year]
test = df[df["Year"] >= cutoff_year].copy()

reg_model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=8, random_state=42)
reg_model.fit(train[feature_cols], train[target_col])

test["Predicted_Peak_MW"] = np.round(reg_model.predict(test[feature_cols]), 2)
test["Abs_Error_MW"] = np.round(np.abs(test[target_col] - test["Predicted_Peak_MW"]), 2)

r2 = r2_score(test[target_col], test["Predicted_Peak_MW"])
mae = mean_absolute_error(test[target_col], test["Predicted_Peak_MW"])
rmse = np.sqrt(mean_squared_error(test[target_col], test["Predicted_Peak_MW"]))
print(f"Test Evaluation -> R2: {r2:.4f} | MAE: {mae:.2f} MW | RMSE: {rmse:.2f} MW")

# Save predictions to Gold table for Power BI
results = test[[
    "Date", "Year", "Month", "Season", "Region", "State_Name",
    "Temp_Max_C", "Daily_Peak_Load_MW", "Predicted_Peak_MW", "Abs_Error_MW"
]].rename(columns={"Daily_Peak_Load_MW": "Actual_Peak_MW"})
results["Date"] = results["Date"].dt.date

spark.createDataFrame(results).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.gold_ml_peak_predictions")

Test Evaluation -> R2: 0.9865 | MAE: 217.32 MW | RMSE: 394.28 MW


In [0]:
import os

export_folder = "/Volumes/workspace/default/australiya_raw_data/powerbi_exports"
os.makedirs(export_folder, exist_ok=True)

tables = [
    "gold_regional_monthly_summary",
    "gold_hourly_demand_profile",
    "gold_daily_peak_demand",
    "gold_ml_peak_predictions"
]

for name in tables:
    spark.table(f"workspace.default.{name}").toPandas().to_csv(f"{export_folder}/{name}.csv", index=False)
    print(f"Saved {name}.csv")

Saved gold_regional_monthly_summary.csv
Saved gold_hourly_demand_profile.csv
Saved gold_daily_peak_demand.csv
Saved gold_ml_peak_predictions.csv
